# Module 5: Perform batch scoring and save predictions to lakehouse

We start with mounting the default lakehouse, as in modules 2-4, and setting configurations to optimize performance, as in module 1.

In [1]:
spark.conf.set("spark.sql.parquet.vorder.enabled", "true") # Enable VOrder write
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true") # Enable automatic delta optimized write

StatementMeta(, 60081eb6-4dcb-49ed-bfe9-5c2a14d3838a, 3, Finished, Available, Finished, False)

#### Read a random sample of cleansed data from lakehouse for the year 2016 and month 3

In [2]:
SEED = 1234 # Random seed
input_df = spark.read.format("delta").load("Tables/nyctaxi_prep")\
            .filter("puYear = 2016 AND puMonth = 3")\
            .sample(True, 0.01, seed=SEED) ## Sampling data to reduce execution time for this tutorial

StatementMeta(, 60081eb6-4dcb-49ed-bfe9-5c2a14d3838a, 4, Finished, Available, Finished, False)

#### Get the trained and registered model to generate predictions

In [3]:
import mlflow
from pyspark.ml.feature import OneHotEncoder, VectorAssembler, StringIndexer
from pyspark.ml import Pipeline
from synapse.ml.core.platform import *
from synapse.ml.lightgbm import LightGBMRegressor

## Define run_uri to fetch the model
run_uri = "runs:/9cd79f2c-6575-4555-899d-4c7664131a25/nyctaxi_tripduration_lightgbm"
loaded_model = mlflow.spark.load_model(run_uri, dfs_tmpdir="Files/tmp/mlflow")

StatementMeta(, 60081eb6-4dcb-49ed-bfe9-5c2a14d3838a, 5, Finished, Available, Finished, False)

/home/trusted-service-user/cluster-env/trident_env/lib/python3.13/site-packages/synapse/ml/core/_proto_compat.py:38: UserWarning: google.protobuf.service module is deprecated. RPC implementations should provide code generator plugins which generate code specific to the RPC implementation. service.py will be removed in Jan 2025
  import google.protobuf.service  # type: ignore  # noqa: F401


Connection pool is full, discarding connection: onelake.blob.fabric.microsoft.com. Connection pool size: 10
Connection pool is full, discarding connection: onelake.blob.fabric.microsoft.com. Connection pool size: 10
Connection pool is full, discarding connection: onelake.blob.fabric.microsoft.com. Connection pool size: 10
Connection pool is full, discarding connection: onelake.blob.fabric.microsoft.com. Connection pool size: 10


2026/10/06 21:19:35 INFO mlflow.spark: File 'runs:/9cd79f2c-6575-4555-899d-4c7664131a25/nyctaxi_tripduration_lightgbm/sparkml' not found on DFS. Will attempt to upload the file.
2026/10/06 21:19:39 INFO mlflow.spark: Copied SparkML model to Files/tmp/mlflow/f9d86979-144d-4d90-bc1d-22245091a4df


StatementMeta(, 60081eb6-4dcb-49ed-bfe9-5c2a14d3838a, 9, Finished, Available, Finished, False)

#### Run model transform on the input dataframe to generate predictions and remove unnecessary vector features created for model training

In [4]:
# Generate predictions by applying model transform on the input dataframe
predictions = loaded_model.transform(input_df)
cols_toremove = ['storeAndFwdFlagIdx', 'timeBinsIdx', 'vendorIDIdx', 'paymentTypeIdx', 'vendorIDEnc',
 'rateCodeIdEnc', 'paymentTypeEnc', 'weekDayEnc', 'pickupHourEnc', 'storeAndFwdFlagEnc', 'timeBinsEnc', 'features','weekDayNameIdx',
 'pickupHourIdx', 'rateCodeIdIdx', 'weekDayNameEnc']
output_df = predictions.withColumnRenamed("prediction", "predictedtripDuration").drop(*cols_toremove)

StatementMeta(, 60081eb6-4dcb-49ed-bfe9-5c2a14d3838a, 6, Finished, Available, Finished, False)

#### Save predictions to lakehouse delta table

In [5]:
table_name = "nyctaxi_pred"
output_df.write.mode("overwrite").format("delta").save(f"Tables/{table_name}")
print(f"Output Predictions saved to delta table: {table_name}")

StatementMeta(, 60081eb6-4dcb-49ed-bfe9-5c2a14d3838a, 7, Finished, Available, Finished, False)

Output Predictions saved to delta table: nyctaxi_pred


#### Preview predicted dataframe

In [6]:
%%sql
SELECT * FROM nyctaxi_pred LIMIT 20

StatementMeta(, 60081eb6-4dcb-49ed-bfe9-5c2a14d3838a, 8, Finished, Available, Finished, False)

<Spark SQL result set with 20 rows and 31 fields>